# Record and replay Responses API tool-calling loops

Agents built on the [Responses API](https://platform.openai.com/docs/api-reference/responses) call tools in a loop: the model asks for a function, your code runs it, you send the result back, and the model decides what to do next. That loop is the hardest part of an agent to test. Every run costs tokens, takes seconds, and can take a slightly different path. Tools that page people, send emails, or move money cannot be re-run casually. Most teams end up with no automated tests of the loop at all.

This cookbook shows a pattern borrowed from HTTP testing libraries such as VCR and Polly: **record once, replay forever**.

1. Run the agent against the live API once and record every `responses.create` request and response to a *cassette* file.
2. Replay the cassette offline. The agent loop, the tool implementations, and your assertions all run for real, but no request leaves the machine and no API key is needed.
3. When a prompt, tool schema, or model change breaks the recording, the replayer explains exactly which request diverged so you can decide whether to re-record.
4. Diff two recordings to see how a change altered the tool calls the model made.
5. Turn a cassette into a deterministic test that runs in CI in milliseconds.

The example domain is incident triage: an SRE assistant that checks service health, inspects recent deploys, and pages an on-call team. The paging tool has a side effect, which is exactly the kind of tool you never want a test suite to trigger for real.

## Prerequisites

- Python 3.10 or newer (the code uses `X | None` type annotations).
- `pip install -r requirements.txt` (the `openai` package).
- An `OPENAI_API_KEY` environment variable **only for the recording step**. Replay, drift detection, and the test run with no key and no network.

## How to run this notebook

Every cell that calls the live API is gated behind `RUN_LIVE`. With the default (`RUN_LIVE` unset or `false`) you can "Run All" with no API key: the recording cells print a note and skip, and the replay cells run against whatever cassettes exist in the `cassettes/` folder next to this notebook. To record fresh cassettes, run the notebook with `RUN_LIVE=true` in the environment.

## Step 1: Setup

The default model is `gpt-5.6-sol`. Any model that supports function calling on the Responses API works; the cassette stores the model name with every request, so a later model change is reported as a mismatch rather than silently replayed.

In [1]:
import difflib
import hashlib
import json
import os
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

from openai import OpenAI
from openai.types.responses import Response

MODEL = "gpt-5.6-sol"

# Set RUN_LIVE=true in the environment to make real API calls and record cassettes.
# With the default of false, "Run All" makes no network calls and needs no API key.
RUN_LIVE = os.getenv("RUN_LIVE", "false").strip().lower() in {"1", "true", "yes"}

CASSETTE_DIR = Path("cassettes")

live_client: OpenAI | None = OpenAI(api_key=os.getenv("OPENAI_API_KEY")) if RUN_LIVE else None

print(f"RUN_LIVE={RUN_LIVE}, model={MODEL}, cassette folder: {CASSETTE_DIR}/")

RUN_LIVE=False, model=gpt-5.6-sol, cassette folder: cassettes/


## Step 2: Define the triage tools

The three tools below are deliberately in-memory so the notebook has no external dependencies. `get_service_status` and `list_recent_deploys` are read-only. `page_oncall` has a side effect: it appends to `PAGES_SENT`, standing in for a real PagerDuty or Opsgenie call.

One rule matters for replay: **tool results must be deterministic for the same arguments**. The replayer matches each request to the recording by hashing the full request, and the request includes every earlier tool result. A tool that returns a timestamp or a random receipt id would change the hash on every run. That is why `page_oncall` derives its receipt id from the arguments rather than from `uuid4()`.

In [2]:
SERVICE_STATUS: dict[str, dict[str, Any]] = {
    "checkout-api": {"status": "degraded", "error_rate": 0.18, "p95_latency_ms": 2400},
    "search-api": {"status": "healthy", "error_rate": 0.002, "p95_latency_ms": 310},
    "payments-worker": {"status": "healthy", "error_rate": 0.0, "p95_latency_ms": 95},
}

RECENT_DEPLOYS: dict[str, list[dict[str, Any]]] = {
    "checkout-api": [
        {"version": "2026.10.13-3", "minutes_ago": 22, "summary": "Enable new tax calculation path"},
        {"version": "2026.10.13-2", "minutes_ago": 190, "summary": "Bump logging library"},
    ],
    "search-api": [
        {"version": "2026.10.12-1", "minutes_ago": 1440, "summary": "Index schema migration"},
    ],
    "payments-worker": [],
}

OWNING_TEAM: dict[str, str] = {
    "checkout-api": "checkout-oncall",
    "search-api": "search-oncall",
    "payments-worker": "payments-oncall",
}

# Side-effect log. In production this would be a call to your paging provider.
PAGES_SENT: list[dict[str, str]] = []


def get_service_status(service: str) -> dict[str, Any]:
    """Return the current health metrics for a service, or an error if it is unknown."""
    status = SERVICE_STATUS.get(service)
    if status is None:
        return {"error": f"unknown service '{service}'", "known_services": sorted(SERVICE_STATUS)}
    return {"service": service, "owning_team": OWNING_TEAM[service], **status}


def list_recent_deploys(service: str) -> dict[str, Any]:
    """Return the most recent deploys for a service, newest first."""
    deploys = RECENT_DEPLOYS.get(service)
    if deploys is None:
        return {"error": f"unknown service '{service}'", "known_services": sorted(RECENT_DEPLOYS)}
    return {"service": service, "deploys": deploys}


def page_oncall(team: str, message: str) -> dict[str, Any]:
    """Page an on-call team. The receipt id is derived from the arguments so replays stay deterministic."""
    receipt_id = hashlib.sha256(f"{team}\n{message}".encode()).hexdigest()[:12]
    PAGES_SENT.append({"team": team, "message": message, "receipt_id": receipt_id})
    return {"paged": True, "team": team, "receipt_id": receipt_id}


TOOL_IMPLEMENTATIONS: dict[str, Callable[..., dict[str, Any]]] = {
    "get_service_status": get_service_status,
    "list_recent_deploys": list_recent_deploys,
    "page_oncall": page_oncall,
}

TOOLS: list[dict[str, Any]] = [
    {
        "type": "function",
        "name": "get_service_status",
        "description": "Get the current health status, error rate, and p95 latency of a service.",
        "parameters": {
            "type": "object",
            "properties": {"service": {"type": "string", "description": "Service name, for example checkout-api."}},
            "required": ["service"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "list_recent_deploys",
        "description": "List the most recent deploys of a service, newest first.",
        "parameters": {
            "type": "object",
            "properties": {"service": {"type": "string", "description": "Service name, for example checkout-api."}},
            "required": ["service"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "page_oncall",
        "description": "Page an on-call team. Only use this when a service is degraded or down.",
        "parameters": {
            "type": "object",
            "properties": {
                "team": {"type": "string", "description": "The owning team returned by get_service_status."},
                "message": {"type": "string", "description": "What is wrong and which deploy is suspected."},
            },
            "required": ["team", "message"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

print(f"{len(TOOLS)} tools defined: {[tool['name'] for tool in TOOLS]}")

3 tools defined: ['get_service_status', 'list_recent_deploys', 'page_oncall']


## Step 3: Write the agent loop against a minimal client interface

The loop below is a standard Responses API function-calling loop. It keeps a manual `input_items` list, appends each response's output items, runs the requested tools, and appends a `function_call_output` item for each call. It stops when the model returns no function calls.

The only unusual thing is the first parameter. Instead of requiring an `OpenAI` instance, `run_triage` accepts anything with a `responses.create(**kwargs)` method. That is the whole seam this pattern needs: the recording client and the replaying client both satisfy it, so the loop itself never changes between live runs and tests.

In [3]:
INSTRUCTIONS = (
    "You are an SRE triage assistant. Investigate the services mentioned in the request using the tools. "
    "Check service status first, then recent deploys for any service that is not healthy. "
    "Page the owning on-call team only when a service is degraded or down, and name the suspected deploy "
    "version in the page message. Finish with a two-sentence summary of what you found and what you did."
)


@dataclass
class TriageRun:
    """The observable result of one agent run: the final answer plus every tool call in order."""

    final_text: str
    tool_calls: list[dict[str, Any]] = field(default_factory=list)
    model_calls: int = 0


def run_triage(
    client: Any,
    question: str,
    instructions: str = INSTRUCTIONS,
    max_model_calls: int = 8,
) -> TriageRun:
    """Run the triage agent loop with any client that exposes responses.create(**kwargs)."""
    input_items: list[Any] = [{"role": "user", "content": question}]
    tool_calls: list[dict[str, Any]] = []

    for model_call in range(1, max_model_calls + 1):
        response = client.responses.create(
            model=MODEL,
            instructions=instructions,
            input=input_items,
            tools=TOOLS,
        )
        function_calls = [item for item in response.output if item.type == "function_call"]
        if not function_calls:
            return TriageRun(final_text=response.output_text, tool_calls=tool_calls, model_calls=model_call)

        # Carry the model's output items (including reasoning items) forward, then add each tool result.
        input_items.extend(response.output)
        for call in function_calls:
            arguments = json.loads(call.arguments)
            result = TOOL_IMPLEMENTATIONS[call.name](**arguments)
            tool_calls.append({"name": call.name, "arguments": arguments, "result": result})
            input_items.append(
                {"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result)}
            )

    raise RuntimeError(f"Agent did not finish within {max_model_calls} model calls")

## Step 4: Fingerprint requests and build the recorder

A cassette is a JSON Lines file with one entry per `responses.create` call. Each entry stores:

- `step`: the position of the call within the run.
- `fingerprint`: a SHA-256 hash of the canonical JSON form of the request (model, instructions, input items, and tools).
- `request`: the request itself, so a mismatch can be explained later.
- `response`: the full `Response` object as returned by `response.model_dump()`.
- `recorded_at`: when it was captured. This is informational and not part of the fingerprint.

Two details make the fingerprint stable across record and replay:

1. Output items the SDK returns are Pydantic models, while the items your code appends are plain dicts. `to_jsonable` normalises both to dicts before hashing.
2. Replayed responses are rebuilt with `Response.model_validate`, so their output items carry the exact ids (`call_id`, `id`) that were recorded. When the loop appends them to `input_items`, the next request hashes to the same value it had during recording.

In [4]:
def to_jsonable(value: Any) -> Any:
    """Convert SDK models and nested containers into plain JSON-serialisable data."""
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json", exclude_none=True)
    if isinstance(value, dict):
        return {key: to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    return value


def canonical_json(value: Any) -> str:
    """Serialise a request deterministically so equal requests hash equally."""
    return json.dumps(to_jsonable(value), sort_keys=True, separators=(",", ":"))


def fingerprint_request(request: dict[str, Any]) -> str:
    """Return a short SHA-256 fingerprint of a responses.create request."""
    return hashlib.sha256(canonical_json(request).encode()).hexdigest()[:16]


class RecordingResponses:
    """Forward responses.create to the live client and append each exchange to a cassette."""

    def __init__(self, live_client: OpenAI, cassette_path: Path) -> None:
        self._live_client = live_client
        self._cassette_path = cassette_path
        self._step = 0

    def create(self, **request: Any) -> Response:
        response = self._live_client.responses.create(**request)
        entry = {
            "step": self._step,
            "fingerprint": fingerprint_request(request),
            "request": to_jsonable(request),
            "response": response.model_dump(mode="json"),
            "recorded_at": datetime.now(timezone.utc).isoformat(),
        }
        with self._cassette_path.open("a", encoding="utf-8") as cassette:
            cassette.write(json.dumps(entry) + "\n")
        self._step += 1
        return response


class RecordingClient:
    """Drop-in replacement for OpenAI that records every responses.create call to a cassette."""

    def __init__(self, live_client: OpenAI, cassette_path: Path) -> None:
        cassette_path.parent.mkdir(parents=True, exist_ok=True)
        cassette_path.unlink(missing_ok=True)
        self.responses = RecordingResponses(live_client, cassette_path)

## Step 5: Build the replayer

The replayer walks the cassette in order. For each `responses.create` call it fingerprints the incoming request and compares it with the recorded one. On a match it returns the recorded response rebuilt as a real `Response` object, so the agent loop cannot tell the difference. On a mismatch it raises `CassetteMismatch` with a unified diff of the two requests. The diff is what makes the pattern maintainable: when a prompt tweak breaks a recording, the error tells you exactly which field changed instead of leaving you to guess.

In [5]:
class CassetteMismatch(Exception):
    """Raised when a replayed request does not match the recorded one at the same step."""


def load_cassette(cassette_path: Path) -> list[dict[str, Any]]:
    """Read a JSON Lines cassette into a list of entries."""
    with cassette_path.open(encoding="utf-8") as cassette:
        return [json.loads(line) for line in cassette if line.strip()]


def describe_mismatch(entry: dict[str, Any], request: dict[str, Any], max_lines: int = 40) -> str:
    """Build a readable explanation of how an incoming request differs from a recorded one."""
    recorded = json.dumps(entry["request"], indent=2, sort_keys=True).splitlines()
    incoming = json.dumps(to_jsonable(request), indent=2, sort_keys=True).splitlines()
    diff = list(difflib.unified_diff(recorded, incoming, "recorded", "incoming", lineterm="", n=1))
    shown = "\n".join(diff[:max_lines])
    if len(diff) > max_lines:
        shown += f"\n... {len(diff) - max_lines} more diff lines"
    return (
        f"Request at step {entry['step']} does not match the cassette "
        f"(recorded {entry['fingerprint']}, incoming {fingerprint_request(request)}).\n{shown}"
    )


class ReplayingResponses:
    """Serve responses.create from a cassette, verifying each request matches the recording."""

    def __init__(self, entries: list[dict[str, Any]]) -> None:
        self._entries = entries
        self.served = 0

    def create(self, **request: Any) -> Response:
        if self.served >= len(self._entries):
            raise CassetteMismatch(
                f"Cassette has {len(self._entries)} entries but the agent made call number {self.served + 1}."
            )
        entry = self._entries[self.served]
        if fingerprint_request(request) != entry["fingerprint"]:
            raise CassetteMismatch(describe_mismatch(entry, request))
        self.served += 1
        return Response.model_validate(entry["response"])


class ReplayingClient:
    """Drop-in replacement for OpenAI that answers responses.create from a cassette with no network."""

    def __init__(self, cassette_path: Path) -> None:
        self.entries = load_cassette(cassette_path)
        self.responses = ReplayingResponses(self.entries)

    def assert_fully_consumed(self) -> None:
        """Fail if the agent made fewer calls than were recorded, which usually means an early exit."""
        if self.responses.served != len(self.entries):
            raise CassetteMismatch(
                f"Only {self.responses.served} of {len(self.entries)} recorded calls were replayed."
            )


def recorded_tool_calls(entries: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Extract the ordered function calls the model made during a recorded run."""
    calls = []
    for entry in entries:
        for item in entry["response"]["output"]:
            if item.get("type") == "function_call":
                calls.append({"name": item["name"], "arguments": json.loads(item["arguments"])})
    return calls


def print_run(label: str, run: TriageRun) -> None:
    """Print a compact summary of an agent run."""
    print(f"{label}: {run.model_calls} model calls, {len(run.tool_calls)} tool calls")
    for index, call in enumerate(run.tool_calls, start=1):
        print(f"  {index}. {call['name']}({json.dumps(call['arguments'])})")
    print(f"  final answer: {run.final_text}")

## Step 6: Record a live run

This is the only cell that spends tokens. It records a triage run for a checkout outage into `cassettes/checkout_outage.jsonl` and prints the path the model took. With `RUN_LIVE` false it prints a note and skips.

In [6]:
OUTAGE_QUESTION = (
    "Customers report checkout failures over the last 30 minutes. "
    "Investigate checkout-api and page the right team if needed."
)
OUTAGE_CASSETTE = CASSETTE_DIR / "checkout_outage.jsonl"

if RUN_LIVE:
    PAGES_SENT.clear()
    live_run = run_triage(RecordingClient(live_client, OUTAGE_CASSETTE), OUTAGE_QUESTION)
    print_run("Live run", live_run)
    print(f"Pages sent for real: {len(PAGES_SENT)}")
    print(f"Recorded {len(load_cassette(OUTAGE_CASSETTE))} model calls to {OUTAGE_CASSETTE}")
else:
    print("RUN_LIVE is false, skipping the live recording.")
    print("Set RUN_LIVE=true and OPENAI_API_KEY to record cassettes/checkout_outage.jsonl.")

RUN_LIVE is false, skipping the live recording.
Set RUN_LIVE=true and OPENAI_API_KEY to record cassettes/checkout_outage.jsonl.


## Step 7: Replay the run offline

Now run the exact same agent loop against the cassette. The tool implementations execute for real (you can see `page_oncall` append to `PAGES_SENT`), the loop reasons over real `Response` objects, and the result matches the live run, but no request leaves the machine. If the cassette has not been recorded yet, this cell explains how to create it.

In [7]:
if OUTAGE_CASSETTE.exists():
    PAGES_SENT.clear()
    replay_client = ReplayingClient(OUTAGE_CASSETTE)
    replayed_run = run_triage(replay_client, OUTAGE_QUESTION)
    replay_client.assert_fully_consumed()
    print_run("Replayed run", replayed_run)
    print(f"Pages that would have been sent: {len(PAGES_SENT)}")
    print(f"Replayed {replay_client.responses.served} recorded model calls with zero network requests.")
else:
    print(f"No cassette at {OUTAGE_CASSETTE}. Run Step 6 with RUN_LIVE=true first, then re-run this cell.")

No cassette at cassettes/checkout_outage.jsonl. Run Step 6 with RUN_LIVE=true first, then re-run this cell.


## Step 8: See what a mismatch looks like

Change the question and replay the same cassette. The very first request no longer matches, and `CassetteMismatch` shows the exact diff. The same error appears when you edit `INSTRUCTIONS`, change a tool description, or switch `MODEL`, which is the signal that the recording is stale and needs to be re-recorded.

In [8]:
if OUTAGE_CASSETTE.exists():
    try:
        run_triage(ReplayingClient(OUTAGE_CASSETTE), OUTAGE_QUESTION + " Also check search-api.")
    except CassetteMismatch as mismatch:
        print(mismatch)
else:
    print(f"No cassette at {OUTAGE_CASSETTE}, nothing to replay yet.")

No cassette at cassettes/checkout_outage.jsonl, nothing to replay yet.


## Step 9: Detect drift between two recordings

Recordings are also a cheap way to see how a change in instructions affects the model's behaviour. The cell below records a second run with a stricter paging policy, then compares the ordered tool calls of the two cassettes. The comparison looks only at tool names and arguments, so it is unaffected by wording changes in the final answer.

`diff_tool_calls` is the building block for a regression check: record a baseline cassette, re-record after a prompt or model change, and fail the build if the agent started or stopped calling a side-effecting tool.

In [9]:
STRICT_INSTRUCTIONS = INSTRUCTIONS + (
    " Paging policy: page only when the error rate is above 25 percent. "
    "Below that threshold, report the degradation in your summary instead of paging."
)
STRICT_CASSETTE = CASSETTE_DIR / "checkout_outage_strict_policy.jsonl"


def diff_tool_calls(baseline: list[dict[str, Any]], candidate: list[dict[str, Any]]) -> list[str]:
    """Compare two recorded runs and describe every difference in their tool-call sequences."""
    baseline_calls = recorded_tool_calls(baseline)
    candidate_calls = recorded_tool_calls(candidate)
    differences: list[str] = []
    for index in range(max(len(baseline_calls), len(candidate_calls))):
        before = baseline_calls[index] if index < len(baseline_calls) else None
        after = candidate_calls[index] if index < len(candidate_calls) else None
        if before == after:
            continue
        before_text = f"{before['name']}({json.dumps(before['arguments'])})" if before else "no call"
        after_text = f"{after['name']}({json.dumps(after['arguments'])})" if after else "no call"
        differences.append(f"call {index + 1}: {before_text} -> {after_text}")
    return differences


if RUN_LIVE:
    PAGES_SENT.clear()
    strict_run = run_triage(RecordingClient(live_client, STRICT_CASSETTE), OUTAGE_QUESTION, STRICT_INSTRUCTIONS)
    print_run("Strict policy run", strict_run)

if OUTAGE_CASSETTE.exists() and STRICT_CASSETTE.exists():
    drift = diff_tool_calls(load_cassette(OUTAGE_CASSETTE), load_cassette(STRICT_CASSETTE))
    if drift:
        print("Tool-call drift between the default and strict policies:")
        for line in drift:
            print(f"  {line}")
    else:
        print("No tool-call drift: both policies made identical tool calls.")
else:
    print(f"Need both {OUTAGE_CASSETTE} and {STRICT_CASSETTE} to diff. Record them with RUN_LIVE=true.")

Need both cassettes/checkout_outage.jsonl and cassettes/checkout_outage_strict_policy.jsonl to diff. Record them with RUN_LIVE=true.


## Step 10: Turn the cassette into a deterministic test

A committed cassette plus the replaying client is a complete unit test of the agent loop. The test below asserts three things that hold for any recording:

- the loop made exactly the recorded tool calls, in order, with the same arguments;
- the side-effecting tool fired exactly as many times during replay as the model requested during recording;
- every recorded model call was consumed, so the loop did not exit early.

It runs here as a plain function. The same function works unchanged under `pytest` because it needs no API key, no network, and no fixtures beyond the cassette file.

In [10]:
def describe_call_differences(expected: list[dict[str, Any]], actual: list[dict[str, Any]]) -> str:
    """Summarise where two tool-call sequences diverge, for assertion messages."""
    for index, (before, after) in enumerate(zip(expected, actual)):
        if before != after:
            return f"call {index + 1}: expected {json.dumps(before)}, got {json.dumps(after)}"
    return f"expected {len(expected)} tool calls, got {len(actual)}"


def test_checkout_outage_replays_exactly() -> None:
    """The triage loop must reproduce the recorded tool calls without any live API access."""
    PAGES_SENT.clear()
    replay_client = ReplayingClient(OUTAGE_CASSETTE)
    run = run_triage(replay_client, OUTAGE_QUESTION)
    replay_client.assert_fully_consumed()

    expected_calls = recorded_tool_calls(replay_client.entries)
    actual_calls = [{"name": call["name"], "arguments": call["arguments"]} for call in run.tool_calls]
    assert actual_calls == expected_calls, describe_call_differences(expected_calls, actual_calls)

    expected_pages = sum(1 for call in expected_calls if call["name"] == "page_oncall")
    assert len(PAGES_SENT) == expected_pages, f"expected {expected_pages} pages, got {len(PAGES_SENT)}"
    assert run.final_text, "the agent must finish with a final answer"


if OUTAGE_CASSETTE.exists():
    test_checkout_outage_replays_exactly()
    print("test_checkout_outage_replays_exactly passed")
else:
    print(f"No cassette at {OUTAGE_CASSETTE}, the test has nothing to replay yet.")

No cassette at cassettes/checkout_outage.jsonl, the test has nothing to replay yet.


To run the same test from a file, copy `run_triage`, the tool definitions, and the replay classes into a module, add the test function to `test_triage.py`, commit the `cassettes/` folder, and run `pytest test_triage.py`. No API key is needed in CI.

## When to re-record

- **The mismatch diff shows a change you intended** (new instructions, a renamed tool, a different model): re-run the recording cells with `RUN_LIVE=true` and commit the new cassette alongside the change.
- **The diff shows a change you did not intend**: the test did its job. Look at the diff before touching the cassette.
- **A tool result changed**: because tool outputs are part of the next request, non-deterministic tool results invalidate the recording. Make the tool deterministic for test inputs, as `page_oncall` does with its receipt id, or exclude the volatile field before the result is appended to `input_items`.

## Limitations and extensions

- **Streaming**: this recorder wraps the non-streaming `responses.create` call. To record streams, collect the final `response.completed` event and store that `Response` object the same way.
- **Server-side state**: the loop sends the full conversation on every call instead of using `previous_response_id`, which keeps each request self-contained and hashable. If you rely on `previous_response_id`, include it in the fingerprint and replay in order.
- **Hosted tools**: built-in tools such as web search or file search run on the server, so their results are part of the recorded response and replay exactly as captured.
- **Sensitive data**: cassettes contain the full request and response. Redact secrets and personal data in `to_jsonable` before writing, and store cassettes with the same care as test fixtures that contain production-like data.
- **Multiple scenarios**: give each scenario its own cassette file and test function. A folder of cassettes becomes an offline evaluation set that you can re-record on a schedule to measure how a new model version changes agent behaviour.